Advanced-RAG : 基于检索增强生成（RAG）的问答系统质量评估
[Query Transformation]
（Decomposition / Abstraction）
        ↓
[Hybrid Retrieval]
 （BM25 + Dense）
        ↓
    [Fusion] 弱排序
     （RRF）
        ↓
    [Rerank] 强排序
        ↓
  [LLM Answer]

    Query Decomposition：问题分解
    Hybrid Retrieval（BM25 + Dense）：retrive阶段，关键词匹配和语义向量检索
    RRF：对候选文档进行混合检索，即Fusion Retrieval，弱排序
    Re-rank重排序：对于召回率低的问题做一个重排序，用Qwen自带的GTE-Rerank，Chroma 检索K=10（海选）-> GTE-Rerank 评分 -> 取前 3 名（精选）

In [17]:
import os
from dotenv import load_dotenv
import dashscope

load_dotenv()
qwen_api_key = os.getenv("DASHSCOPE_API_KEY")
# 输入通义千问 API Key（从阿里云百炼平台获取：https://dashscope.console.aliyun.com/）
if not qwen_api_key:
    raise ValueError("未找到 DASHSCOPE_API_KEY，请检查 .env 文件！")

dashscope.api_key = qwen_api_key
print("API Key 加载成功，准备就绪！")

API Key 加载成功，准备就绪！


In [18]:
import os
from openai import OpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.embeddings import Embeddings
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

# 自定义通义千问 Embedding 类（完善分批处理和错误防护）
class QwenEmbeddings(Embeddings):
    def __init__(self, model="text-embedding-v4", api_key=None, base_url=None, batch_size=10):
        self.model = model
        self.batch_size = batch_size  # 通义千问API强制限制最大10条/批
        self.client = OpenAI(
            api_key=api_key or dashscope.api_key or os.getenv("DASHSCOPE_API_KEY"),
            base_url=base_url or "https://dashscope.aliyuncs.com/compatible-mode/v1"
        )

    def embed_documents(self, texts):
        """批量嵌入文档 - 严格分批处理，过滤空文本"""
        texts = [text.strip() for text in texts if text.strip()]
        if not texts:
            return []

        embeddings = []
        for i in range(0, len(texts), self.batch_size):
            batch_texts = texts[i:i + self.batch_size]
            completion = self.client.embeddings.create(
                model=self.model,
                input=batch_texts
            )
            batch_embeddings = [item.embedding for item in completion.data]
            embeddings.extend(batch_embeddings)

        return embeddings

    def embed_query(self, text):
        """嵌入查询文本 - 处理单个文本"""
        text = text.strip()
        if not text:
            return []

        completion = self.client.embeddings.create(
            model=self.model,
            input=text
        )
        return completion.data[0].embedding

In [19]:
#PDF文档加载器
import os
import requests
import tempfile
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders import PyMuPDFLoader

class PDFLoader(WebBaseLoader):
    def __init__(self, file_path):
        super().__init__(web_path=file_path)
        self.url = file_path

    def load(self):
        """
        利用 Unstructured 架构进行深度解析：可以解析图片、表格、标题等不同元素，并在元数据中保留这些信息
        """
        try:
            print(f"正在启动 PyMuPDF 解析任务: {self.url}")
            response = requests.get(self.url, timeout=15)
            response.raise_for_status()

            temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".pdf")
            temp_file.write(response.content)
            temp_path = temp_file.name
            temp_file.close()  # 重要：写入后立即关闭，释放文件锁

            try:
                # 2. 现在 PyMuPDF 可以自由访问这个文件了
                loader = PyMuPDFLoader(temp_path)
                docs = loader.load()
                print(f"解析完成！共提取 {len(docs)} 页内容。")
            finally:
                # 3. 无论解析是否成功，最后都把临时文件删掉，保持电脑干净
                if os.path.exists(temp_path):
                    os.remove(temp_path)

            return docs

        except Exception as e:
            print(f"PyMuPDF 解析失败: {e}")
            return []
        
pdf_url = "https://arxiv.org/pdf/1706.03762.pdf"
loader = PDFLoader(pdf_url)
docs = loader.load()

if docs:
    print(f"\n--- 文献解析概览 ---")
    print(f"片段总数: {len(docs)}")
        
    for i, doc in enumerate(docs[:3]):
        category = doc.metadata.get("category")
        content_preview = doc.page_content[:50].replace("\n", " ")
        print(f"片段 {i} | 类型: {category} | 内容: {content_preview}...")

正在启动 PyMuPDF 解析任务: https://arxiv.org/pdf/1706.03762.pdf
解析完成！共提取 15 页内容。

--- 文献解析概览 ---
片段总数: 15
片段 0 | 类型: None | 内容: Provided proper attribution is provided, Google he...
片段 1 | 类型: None | 内容: 1 Introduction Recurrent neural networks, long sho...
片段 2 | 类型: None | 内容: Figure 1: The Transformer - model architecture. Th...


In [20]:
import re

def clean_document_content(docs):
    """
    在分割前，对原始页面的文字进行手术级清洗
    """
    cleaned_docs = []
    for doc in docs:
        content = doc.page_content
        # 1. 过滤掉页码 (如 "Page 1 of 12" 或 单独的数字)
        content = re.sub(r'Page \d+ of \d+', '', content)
        # 2. 过滤掉常见的论文页眉干扰
        content = re.sub(r'arXiv:\d+\.\d+v\d+ \[cs\.CL\] \d+ \w+ \d+', '', content)
        # 3. 修复连字符带来的断词
        content = content.replace("-\n", "")
        # 4. 将多个换行符替换为标准分段，去除多余空格
        content = re.sub(r'\n+', '\n', content)
        content = content.strip()
        
        # 只保留有实质内容的页面
        if len(content) > 10:
            doc.page_content = content
            cleaned_docs.append(doc)
            
    print(f"清洗完成：已处理 {len(cleaned_docs)} 页内容")
    return cleaned_docs

cleaned_docs = clean_document_content(docs)

清洗完成：已处理 15 页内容


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def get_text_chunks(docs, chunk_size=600, chunk_overlap=60):
    """
    函数式写法：输入原始文档，返回切割好的块
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", "。", "！", "？", " ", ""]
    )
    
    chunks = splitter.split_documents(docs)
    print(f"已切分为 {len(chunks)} 个片段")
    return chunks

chunks = get_text_chunks(cleaned_docs, chunk_size=600, chunk_overlap=60)

已切分为 171 个片段


In [22]:
from langchain.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

# Embed - 确保使用带分批处理的QwenEmbeddings
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=QwenEmbeddings(
        model="text-embedding-v4",
        api_key=os.environ["DASHSCOPE_API_KEY"],
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
        batch_size=10  # 明确指定批量大小，与API限制一致
    )
)
# 采用 BM25 和 向量检索 的混合策略，提升检索方式的全面性和准确性
bm25_retriever = BM25Retriever.from_documents(chunks) # 初始化 BM25 检索器
bm25_retriever.k = 10
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})  # 修改 向量检索器 K=10
# 组合成混合检索器，并根据RRF把分数进行加权融合，提升整体检索效果
retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.4, 0.6] # 给予向量检索稍高权重
)


In [23]:
# 优化1：Query Decomposition + Rerank 集成流程
import json
import re
import dashscope
from langchain.prompts import ChatPromptTemplate

# ---- 子问题分解 Prompt ----
template ="""你是一个擅长拆解复杂问题的助手。
  将问题分解为 3-5 个互补的子问题：
  1. 每个子问题应能独立检索
  2. 优先提取关键概念（如 d_model、注意力头）
  3. 包含同义词或相关术语
  4. 避免重复

  示例：
  问：Transformer 的学习率调度策略如何工作？
  子问题：
  - Transformer 使用的学习率调度公式是什么
  - warmup steps 参数的作用
  - 学习率如何随训练步数衰减

  问题：{question}
  """

prompt_decomposition = ChatPromptTemplate.from_template(template)

prompt_rag = PromptTemplate(
    input_variables=["context", "question"],
    template="""你是最终答案整合助手。
  基于以下子问题回答，让我们一步步思考，生成一个连贯、简洁的最终答案。

  要求：
  1. 避免重复信息
  2. 保持逻辑连贯
  3. 如果子问题答案矛盾，说明原因
  4. 控制在 3 句话内

  子问题问答：{context}
  最终问题：{question}

  请先思考，然后生成最终回答。
  """
)

llm = ChatOpenAI(
    model_name="qwen-turbo",
    temperature=0,
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=os.environ["DASHSCOPE_API_KEY"],
    max_tokens=512
)

# ---- 辅助函数 ----
def _normalize_query(text: str) -> str:
    """清理编号、符号与多余空白"""
    text = re.sub(r"^\s*(?:[-*•]|\d+[\.)]|[Qq]\d+[::：])\s*", "", str(text))
    text = re.sub(r"\s+", " ", text).strip()
    return text

def _parse_sub_questions(raw_output: str):
    """优先按JSON解析，失败时回退到逐行解析；并做去重与空值过滤"""
    candidates = []
    try:
        parsed = json.loads(raw_output)
        if isinstance(parsed, list):
            candidates = [str(item) for item in parsed]
    except Exception:
        pass
    if not candidates:
        candidates = [line for line in str(raw_output).split("\n") if line.strip()]
    cleaned, seen = [], set()
    for item in candidates:
        normalized = _normalize_query(item)
        if not normalized:
            continue
        key = normalized.lower()
        if key in seen:
            continue
        seen.add(key)
        cleaned.append(normalized)
    return cleaned[:6]

def rerank_documents(query: str, docs: list, top_n: int = 3) -> list:
    """使用 GTE-Rerank-v2 对检索结果重排并取前 top_n 条"""
    if not docs:
        return docs
    doc_texts = [d.page_content for d in docs]
    try:
        response = dashscope.TextReRank.call(
            model="gte-rerank-v2",
            query=query,
            documents=doc_texts,
            top_n=top_n
        )
        if response.status_code == 200:
            return [docs[item.index] for item in response.output.results]
    except Exception as e:
        print(f"⚠️ Rerank 失败，保持原文档顺序：{e}")
    return docs[:top_n]

generate_queries_decomposition = prompt_decomposition | llm | StrOutputParser()

def retrieve_and_rag(question, prompt_rag, sub_question_generator_chain):
    """
    子问题分解 -> 混合检索(K=10) -> 生成子问题答案（不rerank）
    返回：子问题答案、子问题列表、所有检索到的文档（未rerank）
    """
    try:
        raw_sub_questions = sub_question_generator_chain.invoke({"question": question})
        sub_questions = _parse_sub_questions(raw_sub_questions)
    except Exception as e:
        print(f"⚠️ 子问题生成失败，降级为原问题直答：{e}")
        sub_questions = []
    if not sub_questions:
        sub_questions = [question]

    rag_results = []
    all_retrieved_docs = []  # 收集所有检索到的文档（不rerank）
    seen_docs = set()
    rag_chain = prompt_rag | llm | StrOutputParser()

    for sub_question in sub_questions:
        try:
            retrieved_docs = retriever.invoke(sub_question)
            retrieved_docs = [doc for doc in retrieved_docs if getattr(doc, "page_content", "").strip()]
            
            # 收集所有检索到的文档（去重）
            for doc in retrieved_docs:
                text = doc.page_content.strip()
                if text and text not in seen_docs:
                    seen_docs.add(text)
                    all_retrieved_docs.append(doc)

            # 生成子问题答案（使用所有检索到的文档，不rerank）
            context_text = "\n\n".join(doc.page_content for doc in retrieved_docs) if retrieved_docs else "No relevant context found."
            answer = rag_chain.invoke({"context": context_text, "question": sub_question})
        except Exception as e:
            answer = f"子问题处理失败：{e}"

        rag_results.append(answer)

    return rag_results, sub_questions, all_retrieved_docs

question = input("请输入你的问题：")
answers, questions, all_docs = retrieve_and_rag(question, prompt_rag, generate_queries_decomposition)
print(f"\n✅ 子问题生成完成，共检索到 {len(all_docs)} 个文档块（未rerank）")


✅ 子问题生成完成，共检索到 26 个文档块（未rerank）


In [24]:
def format_qa_pairs(questions, answers):
    """Format Q and A pairs"""
    
    formatted_string = ""
    for i, (question, answer) in enumerate(zip(questions, answers), start=1):
        formatted_string += f"Question {i}: {question}\nAnswer {i}: {answer}\n\n"
    return formatted_string.strip()

# ===== 关键修改：在生成最终答案前对所有收集的文档进行 rerank =====
print(f"🔄 正在对 {len(all_docs)} 个文档进行 rerank，选取 top 3...")
reranked_docs = rerank_documents(question, all_docs, top_n=3)
print(f"✅ Rerank 完成，从 {len(all_docs)} 个文档中选出 top 3")

# 使用 rerank 后的文档生成最终答案
context = format_qa_pairs(questions, answers)

# 如果有 rerank 后的文档，将其添加到上下文中
if reranked_docs:
    reranked_context = "\n\n".join([f"参考文档 {i+1}: {doc.page_content}" for i, doc in enumerate(reranked_docs)])
    context = f"{context}\n\n=== 最相关的参考文档（Rerank后）===\n{reranked_context}"

# Prompt
template = """你是最终答案整合助手。
基于以下信息合成一个连贯、简洁的最终答案。

1. 子问题问答对：
{context}

2. 最终问题：{question}

要求：
- 严格基于参考资料回答
- 综合子问题的答案
- 保持逻辑连贯
- 答案简洁明了，控制在 3-5 句话
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    prompt
    | llm
    | StrOutputParser()
)

final_answer = final_rag_chain.invoke({"context": context, "question": question})
print("\n" + "="*80)
print("📝 最终答案：")
print("="*80)
print(final_answer)
print("="*80)

🔄 正在对 26 个文档进行 rerank，选取 top 3...
✅ Rerank 完成，从 26 个文档中选出 top 3

📝 最终答案：
多头注意力通过在不同子空间中并行计算注意力，增强了模型对多样化特征的捕捉能力，避免了单头注意力因信息平均化导致的损失。其结构允许模型同时关注不同位置的信息，提升对复杂依赖关系的建模能力。实验表明，适当数量的注意力头能显著提升模型性能，而单头注意力则表现较差。这种机制在保持计算量不变的情况下优化了模型表达能力。


## Ragas 测评集
基于《Attention Is All You Need》构造 50 个问题及标准答案，用于评测忠诚度、回答相关性、上下文召回率、上下文准确率。

In [29]:
from evaluation_dataset import eval_questions_50, eval_ground_truths_50

def sample_stratified_evaluation(total_samples: int = 5):
    """
    分层采样：确保小样本包含三种题型（与 baseline 完全一致）
    
    题型分布：
    - Q1-38 (76%): 基础知识题 -> 采样约 76% 的样本
    - Q39-45 (14%): 复杂/困难题 -> 采样约 14% 的样本  
    - Q46-50 (10%): 推理题 -> 采样约 10% 的样本
    
    示例：5个样本会采样为 [4基础, 1复杂, 0推理] 或类似分配
    """
    import numpy as np
    
    # 定义题型分组
    basic_indices = list(range(0, 38))  # Q1-38
    complex_indices = list(range(38, 45))  # Q39-45
    reasoning_indices = list(range(45, 50))  # Q46-50
    
    # 按比例分配样本数量
    basic_count = max(1, int(total_samples * 0.76))
    complex_count = max(1, int(total_samples * 0.14))
    reasoning_count = max(1, int(total_samples * 0.10))
    
    # 调整使总和等于 total_samples（处理四舍五入）
    total_allocated = basic_count + complex_count + reasoning_count
    if total_allocated > total_samples:
        if reasoning_count > 0:
            reasoning_count -= (total_allocated - total_samples)
        elif complex_count > 0:
            complex_count -= (total_allocated - total_samples)
        else:
            basic_count -= (total_allocated - total_samples)
    elif total_allocated < total_samples:
        basic_count += (total_samples - total_allocated)
    
    # 随机采样
    np.random.seed(42)  # 固定随机种子，确保可重复性
    sampled_indices = (
        np.random.choice(basic_indices, size=basic_count, replace=False).tolist() +
        np.random.choice(complex_indices, size=complex_count, replace=False).tolist() +
        np.random.choice(reasoning_indices, size=reasoning_count, replace=False).tolist()
    )
    
    print(f"✅ 分层采样完成：共 {total_samples} 题")
    print(f"   - 基础知识题: {basic_count} 题（Q1-38）")
    print(f"   - 复杂/困难题: {complex_count} 题（Q39-45）")
    print(f"   - 推理题: {reasoning_count} 题（Q46-50）")
    print(f"   - 采样题号: {sorted(sampled_indices)}\n")
    
    return sorted(sampled_indices)

# 选择评测题号（分层采样：确保三种类型都有）
eval_indices = sample_stratified_evaluation(total_samples=5)

# 对应的问题和答案
eval_questions_sample = [eval_questions_50[i] for i in eval_indices]
eval_ground_truths_sample = [eval_ground_truths_50[i] for i in eval_indices]

✅ 分层采样完成：共 5 题
   - 基础知识题: 3 题（Q1-38）
   - 复杂/困难题: 1 题（Q39-45）
   - 推理题: 1 题（Q46-50）
   - 采样题号: [4, 33, 36, 43, 49]



In [30]:
# 导入统一的评测集定义
from evaluation_dataset import eval_questions_50, eval_ground_truths_50

# 验证评测集已加载
assert len(eval_questions_50) == 50
assert len(eval_ground_truths_50) == 50
print(f"✅ 已加载 {len(eval_questions_50)} 个问题和 {len(eval_ground_truths_50)} 个标准答案。（包含5个陷阱题/推理题，占比10%）")

✅ 已加载 50 个问题和 50 个标准答案。（包含5个陷阱题/推理题，占比10%）


In [34]:
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import Faithfulness, AnswerRelevancy, LLMContextRecall, LLMContextPrecisionWithReference, AnswerCorrectness

# ====== Ragas 评测：使用 5 项核心指标（与 baseline 保持一致）======
print("="*80)
print("📊 Ragas 评测指标说明")
print("="*80)
print("1. Faithfulness (忠实性): 答案是否忠实于检索到的上下文，无幻觉")
print("2. AnswerRelevancy (答案相关性): 答案是否直接回答了问题")
print("3. ContextRecall (上下文召回率): 检索上下文是否覆盖标准答案的关键信息")
print("4. ContextPrecision (上下文精确度): 检索上下文中有多少是相关的")
print("5. AnswerCorrectness (答案正确性): 答案与标准答案的语义一致程度")
print("="*80 + "\n")

def build_final_answer(main_question: str, sub_questions: list, sub_answers: list, all_retrieved_docs: list) -> tuple:
    """
    将子问题问答对汇总为最终回答（包含 rerank 步骤）
    
    返回：(final_answer, reranked_top3_docs)
    """
    # 步骤1: 对所有收集的文档进行 rerank
    reranked_docs = rerank_documents(main_question, all_retrieved_docs, top_n=3)
    
    # 步骤2: 格式化子问题问答对
    synthesis_context = format_qa_pairs(sub_questions, sub_answers)
    
    # 步骤3: 添加 rerank 后的文档到上下文
    if reranked_docs:
        reranked_context = "\n\n".join([f"参考文档 {i+1}: {doc.page_content}" for i, doc in enumerate(reranked_docs)])
        synthesis_context = f"{synthesis_context}\n\n=== 最相关的参考文档（Rerank后）===\n{reranked_context}"
    
    # 步骤4: 生成最终答案
    synthesis_template = """你是最终答案整合助手。请严格依据下面的子问题回答和参考文档进行整合，不要补充任何外部知识，不要展开背景说明。

子问题问答对：
{context}

最终问题：{question}

要求：
- 如果问题是事实型问题，请输出最直接的结论，尽量 1 句话，最多 2 句话
- 如果子问题回答不足以支持结论，请明确说"我不知道"
- 综合参考文档的信息进行回答
- 保持逻辑连贯

最终回答："""
    synthesis_prompt = ChatPromptTemplate.from_template(synthesis_template)
    synthesis_chain = synthesis_prompt | llm | StrOutputParser()
    final_answer = synthesis_chain.invoke({"context": synthesis_context, "question": main_question})
    
    return final_answer, reranked_docs

def answer_question_for_eval(main_question: str):
    """
    运行完整 Advanced RAG 流程，并返回最终答案与实际使用的检索上下文
    新流程：子问题分解 -> 检索(K=10) -> 生成子问题答案 -> 收集所有文档 -> Rerank(top3) -> 生成最终答案
    """
    sub_answers, sub_questions, all_retrieved_docs = retrieve_and_rag(
        main_question,
        prompt_rag,
        generate_queries_decomposition,
    )
    final_answer, reranked_docs = build_final_answer(main_question, sub_questions, sub_answers, all_retrieved_docs)
    
    # 返回最终答案和 rerank 后的文档（用于评测）
    retrieved_contexts = [doc.page_content for doc in reranked_docs] if reranked_docs else []
    return final_answer, retrieved_contexts

def build_ragas_dataset(eval_questions: list, eval_ground_truths: list, limit: int | None = None, indices: list | None = None):
    """
    构造 Ragas 评测数据集
    
    参数：
    - eval_questions, eval_ground_truths: 问题和回答列表
    - limit: 限制数量（随机采样前 limit 个）
    - indices: 指定题目索引列表（优先级高于 limit）
    """
    # 优先使用 indices，否则用 limit，否则全量
    if indices is not None:
        selected_questions = [eval_questions[i] for i in indices]
        selected_truths = [eval_ground_truths[i] for i in indices]
    elif limit is not None:
        selected_questions = eval_questions[:limit]
        selected_truths = eval_ground_truths[:limit]
    else:
        selected_questions = eval_questions
        selected_truths = eval_ground_truths
    
    records = []
    for idx, (user_input, reference) in enumerate(zip(selected_questions, selected_truths), start=1):
        print(f"[{idx}/{len(selected_questions)}] 正在生成 Advanced RAG 回答与上下文: {user_input}")
        response, retrieved_contexts = answer_question_for_eval(user_input)
        records.append({
            "user_input": user_input,
            "response": response,
            "reference": reference,
            "retrieved_contexts": retrieved_contexts,
        })

    dataset = Dataset.from_list(records)
    return dataset, pd.DataFrame(records)

def run_ragas_evaluation(limit: int | None = None, evaluator_model_name: str = "qwen-max", indices: list | None = None):
    """
    执行 5 项核心 Ragas 指标评测（与 baseline 完全一致）
    
    参数：
    - limit: 随机采样前 limit 个题目
    - evaluator_model_name: 评测模型名称
    - indices: 指定要评测的题目索引列表（优先级高于 limit）
    """
    evaluator_llm = ChatOpenAI(
        model_name=evaluator_model_name,
        temperature=0,
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
        api_key=os.environ["DASHSCOPE_API_KEY"],
        max_tokens=1024,
    )
    evaluator_embeddings = QwenEmbeddings(
        model="text-embedding-v4",
        api_key=os.environ["DASHSCOPE_API_KEY"],
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
        batch_size=10,
    )

    metrics = [
        Faithfulness(llm=evaluator_llm),                    # 忠实性：答案是否基于检索上下文
        AnswerRelevancy(llm=evaluator_llm),                # 答案相关性：答案与问题的相关程度
        LLMContextRecall(llm=evaluator_llm),               # 上下文召回率：检索上下文覆盖标准答案的程度
        LLMContextPrecisionWithReference(llm=evaluator_llm),  # 上下文精确度：检索上下文的相关程度
        AnswerCorrectness(llm=evaluator_llm),              # 答案正确性：答案与标准答案的一致性
    ]

    eval_dataset, eval_records_df = build_ragas_dataset(
        eval_questions_50,
        eval_ground_truths_50,
        limit=limit,
        indices=indices,
    )

    eval_result = evaluate(
        dataset=eval_dataset,
        metrics=metrics,
        llm=evaluator_llm,
        embeddings=evaluator_embeddings,
        raise_exceptions=False,
        show_progress=True,
    )

    score_df = eval_result.to_pandas()
    metric_cols = [
        col
        for col in [
            "faithfulness",
            "answer_relevancy",
            "context_recall",
            "llm_context_precision_with_reference",
            "answer_correctness",  # 新增
        ]
        if col in score_df.columns
    ]
    merged_df = pd.concat([eval_records_df.reset_index(drop=True), score_df[metric_cols].reset_index(drop=True)], axis=1)
    return merged_df

# ========== 小样本评测：使用分层采样（与 baseline 一致）==========
sample_report = run_ragas_evaluation(indices=eval_indices)

# ========== 全量评测：注释掉上面的代码，取消注释下面这行 ==========
# full_report = run_ragas_evaluation()

C:\Users\59118\AppData\Local\Temp\ipykernel_27952\911871246.py:4: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, LLMContextRecall, LLMContextPrecisionWithReference, AnswerCorrectness
C:\Users\59118\AppData\Local\Temp\ipykernel_27952\911871246.py:4: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy, LLMContextRecall, LLMContextPrecisionWithReference, AnswerCorrectness
C:\Users\59118\AppData\Local\Temp\ipykernel_27952\911871246.py:4: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.

📊 Ragas 评测指标说明
1. Faithfulness (忠实性): 答案是否忠实于检索到的上下文，无幻觉
2. AnswerRelevancy (答案相关性): 答案是否直接回答了问题
3. ContextRecall (上下文召回率): 检索上下文是否覆盖标准答案的关键信息
4. ContextPrecision (上下文精确度): 检索上下文中有多少是相关的
5. AnswerCorrectness (答案正确性): 答案与标准答案的语义一致程度

[1/5] 正在生成 Advanced RAG 回答与上下文: Transformer 在每个子层外部使用了哪两种稳定训练的结构？
[2/5] 正在生成 Advanced RAG 回答与上下文: 循环层每层的时间复杂度是多少？
[3/5] 正在生成 Advanced RAG 回答与上下文: 《Attention Is All You Need》主要验证的是哪类任务？
[4/5] 正在生成 Advanced RAG 回答与上下文: 受限自注意力在每层的时间复杂度是多少，相比一般自注意力和循环层各有什么优劣？
[5/5] 正在生成 Advanced RAG 回答与上下文: 如果在序列很长的推理场景下运行 Transformer，相比循环网络，主要的性能瓶颈可能在哪里？


Evaluating:   0%|          | 0/25 [00:00<?, ?it/s]d:\project\CRAG\.venv\Lib\site-packages\langchain_openai\chat_models\base.py:406: UserWarning: Unexpected type for token usage: <class 'NoneType'>
  warnings.warn(f"Unexpected type for token usage: {type(new_usage)}")
Evaluating:   4%|▍         | 1/25 [00:08<03:12,  8.01s/it]d:\project\CRAG\.venv\Lib\site-packages\langchain_openai\chat_models\base.py:406: UserWarning: Unexpected type for token usage: <class 'NoneType'>
  warnings.warn(f"Unexpected type for token usage: {type(new_usage)}")
Evaluating:  24%|██▍       | 6/25 [00:20<00:43,  2.30s/it]d:\project\CRAG\.venv\Lib\site-packages\langchain_openai\chat_models\base.py:406: UserWarning: Unexpected type for token usage: <class 'NoneType'>
  warnings.warn(f"Unexpected type for token usage: {type(new_usage)}")
Evaluating:  28%|██▊       | 7/25 [00:29<01:14,  4.16s/it]d:\project\CRAG\.venv\Lib\site-packages\langchain_openai\chat_models\base.py:406: UserWarning: Unexpected type for token u

In [35]:
# 显示评测结果
metric_cols = [
    col
    for col in [
        "faithfulness",
        "answer_relevancy",
        "context_recall",
        "llm_context_precision_with_reference",
        "answer_correctness",  # 新增：答案正确性
    ]
    if col in sample_report.columns
]

print("="*80)
print("📊 Advanced RAG 评测结果汇总")
print("="*80)
print(f"\n{'指标':<30} {'平均分':>10} {'说明':<40}")
print("-"*80)
print(f"{'Faithfulness (忠实性)':<30} {sample_report['faithfulness'].mean():>10.4f} 答案是否基于检索上下文，无幻觉")
print(f"{'Answer Relevancy (答案相关性)':<30} {sample_report['answer_relevancy'].mean():>10.4f} 答案与问题的相关程度")
print(f"{'Context Recall (上下文召回)':<30} {sample_report['context_recall'].mean():>10.4f} 检索上下文覆盖标准答案的程度")
print(f"{'Context Precision (上下文精度)':<30} {sample_report['llm_context_precision_with_reference'].mean():>10.4f} 检索上下文的相关程度")
print(f"{'Answer Correctness (答案正确性)':<30} {sample_report['answer_correctness'].mean():>10.4f} 答案与标准答案的一致性")
print("-"*80)
print(f"{'总体平均分':<30} {sample_report[metric_cols].mean().mean():>10.4f}")
print("="*80)

# 显示详细结果
diag_cols = [col for col in [
    "user_input",
    "reference",
    "response",
    "faithfulness",
    "answer_relevancy",
    "context_recall",
    "llm_context_precision_with_reference",
    "answer_correctness",  # 新增
 ] if col in sample_report.columns]

print("\n📝 详细结果（前5条）：")
display(sample_report[diag_cols].head(5))

📊 Advanced RAG 评测结果汇总

指标                                    平均分 说明                                      
--------------------------------------------------------------------------------
Faithfulness (忠实性)                 0.4000 答案是否基于检索上下文，无幻觉
Answer Relevancy (答案相关性)           0.8814 答案与问题的相关程度
Context Recall (上下文召回)             0.6000 检索上下文覆盖标准答案的程度
Context Precision (上下文精度)          0.1667 检索上下文的相关程度
Answer Correctness (答案正确性)         0.6786 答案与标准答案的一致性
--------------------------------------------------------------------------------
总体平均分                              0.5453

📝 详细结果（前5条）：


,user_input,reference,response,faithfulness,answer_relevancy,context_recall,llm_context_precision_with_reference,answer_correctness
0,Transformer 在每个子层外部使用了哪两种稳定训练的结构？,每个子层外部都使用残差连接，并在残差相加后进行层归一化。,Transformer 在每个子层外部使用了残差连接（residual connection...,0.0,0.937776,1.0,0.000000,0.817697
1,循环层每层的时间复杂度是多少？,循环层每层的时间复杂度是 O(n 乘 d 的平方)。,循环层每层的时间复杂度是 $O(n \cdot d^2)$。,1.0,0.783182,1.0,0.333333,0.980705
2,《Attention Is All You Need》主要验证的是哪类任务？,论文主要在机器翻译任务上验证模型效果，具体包括 WMT 2014 英德和英法翻译。,《Attention Is All You Need》主要验证的是序列到序列的转换任务，特别...,0.5,0.918195,1.0,0.500000,0.472743
3,受限自注意力在每层的时间复杂度是多少，相比一般自注意力和循环层各有什么优劣？,受限自注意力在每层的时间复杂度是 O(r 乘 n 乘 d)，其中 r 是窗口大小。相比一般自...,受限自注意力在每层的时间复杂度是 $O(r \cdot n \cdot d)$，相比一般自注...,0.0,0.787622,0.0,0.000000,0.773028
4,如果在序列很长的推理场景下运行 Transformer，相比循环网络，主要的性能瓶颈可能在哪里？,主要瓶颈在内存占用。Transformer 需要计算和存储 n×n 的注意力矩阵（n 是序列...,最终回答：在序列很长的推理场景下运行 Transformer，相比循环网络，主要的性能瓶颈可...,0.5,0.980201,0.0,0.000000,0.348817
